# Detecting noisy monitors

This notebook shows how to detect noisy monitors in a dataset using the WhyLabs Monitor Diagnoser. It uses the diagnoser to automatically detect the noisiest monitor for dataset, get a diagnosis of
the conditions causing the noise, get recommended changes and where automatable, apply those changes.

## Install requirements

In [1]:
%pip install .

## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [2]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = 'org-0'
dataset_id = 'model-0'
api_key = getpass.getpass()
api_endpoint = 'https://songbird.development.whylabsdev.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

Initialize the Monitor Diagnoser with the org_id and dataset_id.

In [3]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

## Run the default diagnosis

With no further input, the diagnoser will make a series of calls to identify the noisiest monitor, segment and columns; and then perform a diagnosis.

In [5]:
# for now, we need to enforce this to run using local server
import os
os.environ['USE_LOCAL_SERVER'] = 'server'
monitor_report = diagnoser.diagnose()
monitor_report

MonitorDiagnosisReport(orgId='org-0', datasetId='model-0', analyzerId='adorable-goldenrod-lion-9438-analyzer', interval='2024-03-16T00:00:00.000Z/2024-04-15T00:00:00.000Z', expectedBatchCount=30, diagnosticData=DiagnosticDataSummary(diagnosticSegment=Segment(tags=[]), diagnosticProfile=ProfileSummary(minRowName='issue_d', minRowCount=2494691, maxRowName='issue_d', maxRowCount=2494691), diagnosticBatches=BatchesSummary(minBatchName='issue_d', minBatchCount=30, maxBatchName='issue_d', maxBatchCount=30), analysisResults=AnalysisResultsSummary(results=ResultRecord(diagnosedColumnCount=27, batchCount=30), failures=FailureRecord(totalFailuresCount=0, maxFailuresCount=0, meanFailuresCount=0, byColumnCount=[], byTypeCount=[]), anomalies=AnomalyRecord(totalAnomalyCount=31, maxAnomalyCount=30, meanAnomalyCount=15, batchCount=30, byColumnCount=[('issue_d', 30), ('url', 1)], byColumnBatchCount=[('addr_state', 19), ('application_type', 19), ('debt_settlement_flag', 30), ('desc', 1), ('disbursement_

In [6]:
print(monitor_report.describe())

Diagnosis is for monitor "wrong-drift-crowded-orchid-coyote-2773" [adorable-goldenrod-lion-9438] in model-0 org-0, over interval 2024-03-16T00:00:00.000Z/2024-04-15T00:00:00.000Z.

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "adorable-goldenrod-lion-9438-analyzer" targets 125 columns and ran on 27 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 2494691 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 27 columns and 30 batches.
Found 31 anomalies in 2 columns, with up to 100.0% (30) batches having anomalies per column and 50.0% (15.0) on average.
Columns with anomalies are:
|         |   0 |
|:--------|----:|
| issue_d |  30 |
| url     |   1 |

No failures were detected.

No issues impacting diagnosis quality were detected
Conditions that may contribute to noise include:
	* Condition changing_discr

The monitor report can be serialized to a JSON file for later use.

In [7]:
with open('monitor_report.json', 'w') as f:
    f.write(monitor_report.json())

In [8]:
from whylabs_toolkit.monitor.diagnoser.models import MonitorDiagnosisReport

with open('monitor_report.json', 'r') as f:
    monitor_report = MonitorDiagnosisReport.parse_raw(f.read())
print(monitor_report.json(indent=2))

{
  "orgId": "org-0",
  "datasetId": "model-0",
  "analyzerId": "adorable-goldenrod-lion-9438-analyzer",
  "interval": "2024-03-16T00:00:00.000Z/2024-04-15T00:00:00.000Z",
  "expectedBatchCount": 30,
  "diagnosticData": {
    "diagnosticSegment": {
      "tags": []
    },
    "diagnosticProfile": {
      "minRowName": "issue_d",
      "minRowCount": 2494691,
      "maxRowName": "issue_d",
      "maxRowCount": 2494691
    },
    "diagnosticBatches": {
      "minBatchName": "issue_d",
      "minBatchCount": 30,
      "maxBatchName": "issue_d",
      "maxBatchCount": 30
    },
    "analysisResults": {
      "results": {
        "diagnosedColumnCount": 27,
        "batchCount": 30
      },
      "failures": {
        "totalFailuresCount": 0,
        "maxFailuresCount": 0,
        "meanFailuresCount": 0,
        "byColumnCount": [],
        "byTypeCount": []
      },
      "anomalies": {
        "totalAnomalyCount": 31,
        "maxAnomalyCount": 30,
        "meanAnomalyCount": 15,
        

## Ask for recommended changes

Given the diagnosis report for the monitor, the ChangeRecommender will recommend changes to make to the monitor. By default it will make recommendations for all columns where it has detected noise-related conditions. Set the `min_anomaly_count` property to restrict this to only columns that caused a certain number of anomalies.


In [9]:
from whylabs_toolkit.monitor.diagnoser.recommendation.change_recommender import ChangeRecommender

recommender = ChangeRecommender(monitor_report)
recommender.min_anomaly_count = 1
changes = recommender.recommend()
print('\n'.join([f'{i+1}. {c.describe()}' for i, c in enumerate(changes)]))

1. Remove columns from the analyzer for ['issue_d', 'url']


## Execute automatable changes

A subset of recommended changes can be executed automatically by the recommender. Pass the ones you want to make into the `make_changes` call, or pass all changes if you want it to make all of the automatable changes.

In [10]:
automatable_changes = [c for c in changes if c.can_automate()]
print('\n'.join([c.describe() for c in automatable_changes]))

Remove columns from the analyzer for ['issue_d', 'url']


In [11]:
change_results = recommender.make_changes(automatable_changes)
print(change_results.describe())

Successfully made the following changes:
	* Remove columns from the analyzer for ['issue_d', 'url']


Note that the monitor will still appear to the diagnoser as the noisiest monitor until enough time has passed for the impact of the monitor changes to be observed. You may want to use the WhyLabs preview UI to view what impacts may be expected from the change.

## Reviewing other noisy monitors

The diagnoser can be used to review other noisy monitors in the dataset. The `noisy_monitors` property will return a list of the noisiest monitors, and the `monitor_id_to_diagnose` property can be set to the monitor_id of the monitor to diagnose.

In [12]:
import pandas as pd
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors])
noisy_monitors_df

,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,adorable-goldenrod-lion-9438,adorable-goldenrod-lion-9438-analyzer,frequent_items,2,1,31,30,1,15,0,[]
1,unsightly-orchid-gorilla-4971,unsightly-orchid-gorilla-4971-analyzer,frequent_items,3,1,33,30,1,11,0,[]
2,concerned-skyblue-penguin-6734,concerned-skyblue-penguin-6734-analyzer,frequent_items,3,1,32,30,1,10,0,[]
3,proud-seagreen-carabeef-65,proud-seagreen-carabeef-65-analyzer,histogram,1,1,28,28,28,28,0,[]
4,kind-cyan-kangaroo-1253,kind-cyan-kangaroo-1253-analyzer,histogram,1,1,28,28,28,28,0,[]
...,...,...,...,...,...,...,...,...,...,...,...
93,numerical-drift-monitor-60dfcc,numerical-drift-analyzer-60dfcc,histogram,1,1,2,2,2,2,2,"[email, slack]"
94,stormy-olive-butterfly-8693,stormy-olive-butterfly-8693-analyzer,histogram,1,1,2,2,2,2,0,[]
95,fine-magenta-nightingale-9708,fine-magenta-nightingale-9708-analyzer,unique_est_ratio,26,1,39,2,1,1,0,[]
96,None,eager-violet-newt-4599-analyzer,count_null_ratio,21,1,28,2,1,1,0,[]


In [13]:
diagnoser.monitor_id_to_diagnose = noisy_monitors_df.iloc[1]['monitor_id']
monitor_report = diagnoser.diagnose()
print(monitor_report.describe())

Diagnosis is for monitor "unsightly-orchid-gorilla-4971" [unsightly-orchid-gorilla-4971] in model-0 org-0, over interval 2024-03-16T00:00:00.000Z/2024-04-15T00:00:00.000Z.

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "unsightly-orchid-gorilla-4971-analyzer" targets 30 columns and ran on 26 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 2494691 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 26 columns and 30 batches.
Found 33 anomalies in 3 columns, with up to 100.0% (30) batches having anomalies per column and 36.7% (11.0) on average.
Columns with anomalies are:
|         |   0 |
|:--------|----:|
| issue_d |  30 |
| desc    |   2 |
| url     |   1 |

No failures were detected.

No issues impacting diagnosis quality were detected
Conditions that may contribute to noise include:
	* Condition chan

You can also use the `noisy_monitors_with_actions` property to prioritize noise in monitors with actions, as these are most likely to cause alert fatigue.

In [14]:
pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors_with_actions])


,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,energetic-black-cobra-7838,energetic-black-cobra-7838-analyzer,unique_est,8,1,100,28,3,12,1,[email]
1,frequent-items-drift-monitor-uu0ax8,frequent-items-drift-analyzer-uu0ax8,frequent_items,3,1,31,28,1,10,3,"[email, slack, email-victor-at-whylabs]"
2,old-crimson-starling-2516,old-crimson-starling-2516-analyzer,frequent_items,3,1,31,28,1,10,1,[email]
3,frequent-items-drift-monitor-48ukw1,frequent-items-drift-analyzer-48ukw1,frequent_items,3,1,31,28,1,10,2,"[email, slack]"
4,frequent-items-drift-monitor-jepz7t,frequent-items-drift-analyzer-jepz7t,frequent_items,3,1,31,28,1,10,2,"[email, slack]"
5,frequent-items-drift-monitor-pxexvn,frequent-items-drift-analyzer-pxexvn,frequent_items,3,1,31,28,1,10,2,"[email, slack]"
6,frequent-items-drift-monitor-u31vmb,frequent-items-drift-analyzer-u31vmb,frequent_items,3,1,31,28,1,10,2,"[email, slack]"
7,elated-gray-baboon-4620,elated-gray-baboon-4620-analyzer,count_null_ratio,15,1,70,28,1,4,1,[email]
8,nice-burlywood-tarsier-4771,nice-burlywood-tarsier-4771-analyzer,unique_est,7,1,97,26,3,13,2,"[slack, email]"
9,unique-estimate-ratio-monitor-ccf7cl,unique-estimate-ratio-analyzer-ccf7cl,unique_est_ratio,104,1,358,7,1,3,2,"[email, slack]"
